# Forecasting horario de consumo energético

Comparación reproducible de dos modelos de *gradient boosting* frente a referencias ingenuas. La tarea es predecir **una hora por delante** utilizando exclusivamente información disponible hasta la hora anterior.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from src.data import load_energy_data
from src.evaluation import naive_from_lag, regression_metrics, split_at
from src.features import build_causal_features
from src.modeling import build_catboost, build_xgboost

DATA_PATH = Path('data/energy_train.csv')
TEST_START = '2016-01-01 00:00:00'

## 1. Calidad y preparación de la serie

La carga elimina marcas temporales duplicadas, completa la rejilla horaria e interpola únicamente los huecos internos.

In [ ]:
energy, quality = load_energy_data(DATA_PATH)
quality, energy.describe()

In [ ]:
energy.loc['2015-01-01':'2015-01-14'].plot(figsize=(13, 4), legend=False)
plt.title('Consumo horario — dos semanas de ejemplo')
plt.ylabel('Consumo')
plt.show()

## 2. Variables causales y división temporal

Los *lags* usan observaciones anteriores. Las medias y desviaciones móviles se calculan sobre `Energy.shift(1)`, de modo que el consumo de la hora objetivo nunca participa en sus propias variables. El test es el periodo final de la serie y no se baraja.

In [ ]:
features = build_causal_features(energy)
train, test = split_at(features, TEST_START)
feature_columns = [column for column in features if column != 'target']

print(f'Train: {train.index.min()} — {train.index.max()} ({len(train):,} horas)')
print(f'Test:  {test.index.min()} — {test.index.max()} ({len(test):,} horas)')
features.head()

## 3. Referencias y modelos

Se comparan una persistencia de una hora, una referencia semanal, XGBoost y CatBoost. WAPE expresa el error absoluto total como porcentaje del consumo observado.

In [ ]:
x_train, y_train = train[feature_columns], train['target']
x_test, y_test = test[feature_columns], test['target']
predictions = {
    'Persistence (1h)': naive_from_lag(test, 1),
    'Seasonal naive (168h)': naive_from_lag(test, 168),
}
models = {'XGBoost': build_xgboost(), 'CatBoost': build_catboost()}
for name, model in models.items():
    model.fit(x_train, y_train)
    predictions[name] = model.predict(x_test)
metrics = pd.DataFrame([
    {'model': name, **regression_metrics(y_test, values)}
    for name, values in predictions.items()
]).sort_values('RMSE')
metrics

In [ ]:
best_model = metrics.iloc[0]['model']
comparison = pd.DataFrame({'Real': y_test, best_model: predictions[best_model]}, index=test.index).tail(24 * 7)
comparison.plot(figsize=(13, 5))
plt.title('Predicción una hora por delante — última semana de test')
plt.ylabel('Consumo')
plt.show()

## 4. Interpretación y límites

Los modelos de boosting reducen de forma clara el error frente a las referencias. Esta evaluación representa una operación *rolling* de una hora: al avanzar el reloj, el consumo real anterior ya está disponible. No demuestra rendimiento para un horizonte de días o años, no incorpora temperatura ni festivos y el archivo original no documenta suficientemente la procedencia ni la unidad física del consumo.